# Solutions — Transformations: Normalize, Enrich, Derive, Windows, SCD2

Exercise notebook: `08_data_engineering_python/03_transformations.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Normalize customer records

**Working solution**

In [ ]:
from datetime import datetime

def _norm_date(text):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(text.strip(), fmt).date().isoformat()
        except (ValueError, AttributeError):
            continue
    return None

def normalize_customer(row):
    email = (row.get("email") or "").strip().lower()
    return {
        **row,
        "name": " ".join((row.get("name") or "").split()).title(),
        "email": email or None,
        "country": (row.get("country") or "").strip().upper(),
        "signup_date": _norm_date(row.get("signup_date") or ""),
        "is_active": str(row.get("is_active", "")).strip().lower() == "true",
    }

**Explanation**

`{**row, ...}` returns a new dict overriding just the normalized fields. Trying several `strptime` formats in order is the standard multi-format date parse.

**Complexity:** O(1).

**Common mistakes**

- Mutating the input row in place.
- `.title()` quirk: `"o'brien"` → `"O'Brien"` is fine, but `"mcdonald"` → `"Mcdonald"` — names are hard; know the limits.

## Exercise 2 — 🟡 Medium · Enrich via lookups (hash joins)

**Working solution**

In [ ]:
def enrich_orders(orders, customers, products):
    cust = {c["customer_id"]: c for c in customers}
    prod = {p["product_id"]: p for p in products}
    enriched = []
    for o in orders:
        c = cust.get(o.get("customer_id"), {})
        p = prod.get(o.get("product_id"), {})
        enriched.append({**o, "customer_segment": c.get("segment"), "customer_country": c.get("country"),
                         "product_category": p.get("category") or None, "product_name": p.get("name")})
    return enriched

**Explanation**

This *is* a hash join: build hash tables from the smaller sides, probe with each order. Spark's broadcast join is the distributed version.

**Complexity:** O(n + c + p).

**Common mistakes**

- A nested loop per order over all customers (O(n·c)).

## Exercise 3 — 🟡 Medium · Derived columns

**Working solution**

In [ ]:
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")

def _bucket(amount):
    if amount < 1000:
        return "<1k"
    if amount < 10_000:
        return "1k-10k"
    return "10k+"

def derive(order):
    local = order["order_ts"].astimezone(IST)
    return {**order, "order_date": order["order_ts"].date().isoformat(), "order_hour_ist": local.hour,
            "weekday": local.strftime("%A"), "is_weekend": local.weekday() >= 5, "amount_bucket": _bucket(order["amount"])}

**Explanation**

`zoneinfo` (stdlib since 3.9) handles time zones; convert with `astimezone`. Business calendars (weekends, 'which day did this sale happen') depend on local time, storage uses UTC.

**Complexity:** O(1).

**Common mistakes**

- Deriving the weekday from UTC for an Indian business — a sale at 01:00 IST Saturday is still Friday in UTC.
- On Windows, `zoneinfo` needs the `tzdata` package (it's in requirements.txt).

## Exercise 4 — 🟠 Hard · Window functions in Python

**Working solution**

In [ ]:
from itertools import groupby
from operator import itemgetter

def add_window_columns(rows, partition_by, order_by, value):
    ordered = sorted(rows, key=itemgetter(partition_by, order_by))
    out = []
    for _, group in groupby(ordered, key=itemgetter(partition_by)):
        running, previous = 0.0, None
        for n, row in enumerate(group, start=1):
            running += row[value]
            out.append({**row, "row_number": n, "running_total": round(running, 2), "prev_value": previous})
            previous = row[value]
    return out

**Explanation**

`sorted` + `itertools.groupby` is the pattern for "process each partition in order" — groupby needs the input sorted by the same key. Python's stable sort keeps ties in input order.

**Complexity:** O(n log n).

**Common mistakes**

- Using `groupby` on unsorted data (partitions split into many groups).

## Challenge — 🔴 Very Hard · Slowly Changing Dimension Type 2

**Working solution**

In [ ]:
import copy

def apply_scd2(dimension, updates, key, tracked, effective_date):
    seen = set()
    for u in updates:
        if u[key] in seen:
            raise ValueError(f"duplicate key in updates: {u[key]!r}")
        seen.add(u[key])
    result = copy.deepcopy(dimension)
    current = {row[key]: row for row in result if row["is_current"]}
    meta = {"valid_from", "valid_to", "is_current"}
    for update in updates:
        k = update[key]
        attrs = {f: v for f, v in update.items() if f not in meta}
        existing = current.get(k)
        if existing is None:
            result.append({**attrs, "valid_from": effective_date, "valid_to": None, "is_current": True})
            continue
        if any(existing.get(f) != attrs.get(f) for f in tracked):
            existing["valid_to"], existing["is_current"] = effective_date, False
            result.append({**attrs, "valid_from": effective_date, "valid_to": None, "is_current": True})
        else:
            existing.update({f: v for f, v in attrs.items() if f not in tracked})
    return sorted(result, key=lambda r: (r[key], r["valid_from"]))

**Explanation**

SCD2 keeps history: a customer moving from Hyderabad to Vijayawada produces two rows, so last year's sales still roll up to the old city. Indexing current versions by key
turns the merge into one pass. In Spark/Delta Lake you'd express this as a `MERGE INTO`; the decision table (new / changed tracked / changed untracked / unchanged) is the same.

**Complexity:** O(d + u + total log total).

**Common mistakes**

- Overwriting the current row (Type 1) when history is required.
- Closing versions but forgetting `is_current = False`.
- Not being idempotent on rerun.